# Projected Non-Reversible Anchored Langevin on the MAGIC Gamma Telescope data

This notebook runs the complete experiment end to end: data loading, the smooth
anchor, the smoothed $\ell_p$ constraint, the manuscript-faithful skew-symmetric
matrix $J(w)$, the projected non-reversible anchored Langevin sampler, the
automatic checks, the diagnostics, the sensitivity studies, the posterior
predictive evaluation and every figure and table.

**The target.** For labels $y_i\in\{0,1\}$ and $z = Xw$,

$$U(w)=\sum_i\big[\mathrm{softplus}(z_i)-y_iz_i\big]+\frac{w_0^2}{2\sigma_{\text{intercept}}^2}+\lambda_{\text{lasso}}\sum_{j\ge1}|w_j|,$$

with the smooth anchor $U_0$ obtained by replacing $|w_j|$ with
$\sqrt{w_j^2+\delta_{\text{anchor}}^2}$, so that
$\log a(w)=U(w)-U_0(w)=\lambda_{\text{lasso}}\sum_{j\ge1}\big(|w_j|-\sqrt{w_j^2+\delta_{\text{anchor}}^2}\big)\le 0$.

**The update.**

$$w_{k+1}=\Pi_K\Big[w_k-h\,a_k\,(I_d+\alpha J_k)\nabla U_0(w_k)+\sqrt{2ha_k}\,\xi_k\Big],\qquad \xi_k\sim N(0,I_d).$$

**The constraint.** $K=\{w: g(w)\le\Lambda_{\text{constraint}}\}$ with
$g(w)=\sum_i (w_i^2+\epsilon_{\text{constraint}}^2)^{p_{\text{constraint}}/2}$.
The constrained posterior is the **truncation** of $\exp(-U)$ to $K$.

> **Data.** Place the UCI file `magic04.data` in `data/` (or set `DATA_PATH`
> below). If it is absent you may set `USE_SYNTHETIC = True` to exercise the
> code path on a clearly-labelled surrogate — that surrogate is **not** the
> MAGIC data and supports no scientific claim.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the ``pnral`` package importable no matter where the kernel started:
# walk upwards until the package directory is found.
_root = Path.cwd().resolve()
for _candidate in (_root, *_root.parents):
    if (_candidate / "pnral" / "__init__.py").is_file():
        sys.path.insert(0, str(_candidate))
        os.chdir(_candidate)          # relative paths (data/, results/) resolve here
        break
else:
    raise RuntimeError("could not locate the 'pnral' package; run this notebook "
                       "from inside the repository")

import pnral
from pnral import anchor, constraint, diagnostics as diag, nonreversible_matrix as nrm
from pnral import prediction as pred, projection, sampler
from pnral.config import ExperimentConfig
from pnral.data import load_dataset, save_preprocessing
from pnral.experiment import run_experiment
from pnral.target import LogisticTarget

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
diag.apply_style()
print("pnral", pnral.__version__, "| numpy", np.__version__, "| arviz", az.__version__)

## 1. Configuration

Every parameter of the study lives in one place and is serialised with the run.

In [ ]:
DATA_PATH = "data/magic04.data"       # <- the local UCI file
USE_SYNTHETIC = not Path(DATA_PATH).is_file()   # surrogate only if the file is missing
OUTPUT_DIR = "results_notebook"

config = ExperimentConfig()
config.data.data_path = DATA_PATH
config.data.use_synthetic = USE_SYNTHETIC
config.output_dir = OUTPUT_DIR

# Reduce these for a fast pass; the defaults below are the production settings.
config.sampler.n_iterations = 20000
config.sampler.burn_in = 5000
config.sampler.number_of_chains = 4
config.sampler.thinning = 1
config.sampler.alphas = (0.0, 0.1, 0.25, 0.5, 1.0)

if USE_SYNTHETIC:
    print("!" * 78)
    print("magic04.data not found -> SYNTHETIC SURROGATE. Results are not science.")
    print("!" * 78)
print(json.dumps(config.to_dict(), indent=2)[:900], "...")

## 2. Data

Stratified 80/20 split, `StandardScaler` fitted on the training predictors only, intercept column appended afterwards. `d` is read off the design matrix.

In [ ]:
data = load_dataset(config.data.data_path, test_size=config.data.test_size,
                    split_seed=config.data.split_seed,
                    use_synthetic=config.data.use_synthetic,
                    synthetic_n_rows=config.data.synthetic_n_rows,
                    synthetic_seed=config.data.synthetic_seed)
d = data.d
print(f"train {data.n_train} x {d} | test {data.n_test} | d = {d}")
print("parameters:", data.parameter_names)
print(f"labels: g -> 1, h -> 0; training gamma fraction {data.y_train.mean():.4f}")
assert d == 11, "the MAGIC design should give d = 11"
pd.DataFrame(data.X_train[:5], columns=data.parameter_names)

## 3. Target, anchor and the anchoring coefficient

$\log a$ is evaluated directly from its closed form, never as $\log(e^{U}/e^{U_0})$, and must satisfy $-10\lambda_{\text{lasso}}\delta_{\text{anchor}}\le\log a\le 0$.

In [ ]:
target = LogisticTarget(data.X_train, data.y_train,
                        lambda_lasso=config.target.lambda_lasso,
                        sigma_intercept=config.target.sigma_intercept,
                        delta_anchor=config.target.delta_anchor)

w_map = sampler.smooth_map(target)                     # L-BFGS-B on U0
U0_map, gradient = anchor.value_and_grad_U0(target, w_map)
log_a_min, log_a_max, a_min, a_max = anchor.anchor_bounds(target)

print(f"U(w_MAP)      = {target.potential_U(w_map):.6f}")
print(f"U0(w_MAP)     = {U0_map:.6f}")
print(f"log a(w_MAP)  = {anchor.log_anchor_coefficient(target, w_map):.6f}"
      f"   (bounds [{log_a_min:g}, {log_a_max:g}])")
print(f"a(w_MAP)      = {anchor.anchor_coefficient(target, w_map):.6f}"
      f"   (bounds [{a_min:.3g}, {a_max:g}])")
print(f"||grad U0||   = {np.linalg.norm(gradient):.6e}")
print(f"U - U0 - log a = {target.potential_U(w_map) - U0_map - anchor.log_anchor_coefficient(target, w_map):.3e}")
print(f"L guide (0.25 ||X||_2^2) = {target.likelihood_smoothness_guide():.6g}")
anchor.check_anchor_bounds(target, w_map)
pd.Series(w_map, index=data.parameter_names, name="smooth MAP").to_frame().T

## 4. The smoothed $\ell_p$ constraint and the pilot rule for $\Lambda_{\text{constraint}}$

An unconstrained reversible ($\alpha=0$) anchored pilot chain at a conservative step size supplies the empirical 0.999 quantile of $g$; the threshold is then frozen.

In [ ]:
p_constraint = config.constraint.p_constraint
epsilon_constraint = config.constraint.epsilon_constraint
g_min = constraint.g_minimum(d, p_constraint, epsilon_constraint)
print(f"g(0) = d * eps^p = {g_min:.8g}  (check: "
      f"{constraint.g_constraint(np.zeros(d), p_constraint, epsilon_constraint):.8g})")

j_spec = sampler.JSpec.create(config.sampler.triples_mode, d, p_constraint,
                              epsilon_constraint, config.sampler.swirl_scale)
pilot_step = config.constraint.pilot_step_size_factor / target.likelihood_smoothness_guide()
pilot = sampler.run_chain(target=target, w0=w_map, alpha=0.0, step_size=pilot_step,
                          n_iterations=config.constraint.pilot_iterations,
                          seed=config.constraint.pilot_seed, j_spec=j_spec,
                          Lambda_constraint=None, p_constraint=p_constraint,
                          epsilon_constraint=epsilon_constraint,
                          burn_in=config.constraint.pilot_burn_in, constrained=False)
pilot_g = constraint.g_constraint(pilot.retained, p_constraint, epsilon_constraint)
selection = constraint.choose_threshold_from_pilot(
    pilot_g, d, p_constraint, epsilon_constraint,
    quantile=config.constraint.pilot_quantile, inflation=config.constraint.pilot_inflation)
Lambda_constraint = selection.Lambda_constraint

print(f"pilot g: min {pilot_g.min():.5f}, median {np.median(pilot_g):.5f}, "
      f"q0.999 {selection.pilot_quantile_value:.5f}, max {pilot_g.max():.5f}")
print(f"Lambda_constraint = {Lambda_constraint:.6f}  (FROZEN, identical for every alpha)")
print(f"unconstrained pilot samples with g > Lambda: {selection.exceedance_fraction:.4%}")
print(f"radii for the sensitivity study: {selection.Lambda_small:.5f} / "
      f"{Lambda_constraint:.5f} / {selection.Lambda_large:.5f}")
print("\nThe constrained posterior is the TRUNCATION of exp(-U) to "
      "K = {w : g(w) <= Lambda_constraint}.")
constraint.validate_threshold(Lambda_constraint, d, p_constraint, epsilon_constraint);

## 5. The non-reversible matrix $J(w)$

Disjoint triples $(0,1,2),(3,4,5),(6,7,8)$ for the primary experiment; coordinates 9 and 10 receive reversible drift and diffusion but no direct swirl. We verify skew symmetry, $\operatorname{div}J=0$ by centred differences, and tangency $J(w)n(w)=0$ on the boundary.

In [ ]:
from scipy.optimize import brentq

rng = np.random.default_rng(0)
rows = []
for mode in ("disjoint", "overlapping"):
    builder, triples, block_scale = nrm.make_J_builder(mode, d, p_constraint,
                                                       epsilon_constraint,
                                                       config.sampler.swirl_scale)
    probes = [w_map + 0.4 * rng.standard_normal(d) for _ in range(8)]
    skew = max(nrm.check_skew_symmetry(builder(w)) for w in probes)
    divergence = max(nrm.check_divergence_free(builder, w, tolerance=np.inf)
                     for w in probes)
    tangency = []
    for _ in range(8):                                  # exact boundary points
        direction = rng.standard_normal(d)
        scale = brentq(lambda t: constraint.g_constraint(t * direction, p_constraint,
                                                         epsilon_constraint)
                       - Lambda_constraint, 1e-8, 1e4)
        boundary = scale * direction
        normal = constraint.outward_normal(boundary, p_constraint, epsilon_constraint)
        tangency.append(nrm.check_boundary_tangency(builder(boundary), normal,
                                                    tolerance=np.inf))
    rows.append({"mode": mode, "triples": triples, "block_scale": block_scale,
                 "max |J + J^T|": skew, "max |div J|": divergence,
                 "max ||J n||_inf on the boundary": max(tangency)})
display(pd.DataFrame(rows))

J = nrm.construct_J_disjoint(w_map, p_constraint, epsilon_constraint, 1.0)
print("rows 9 and 10 carry no swirl:", np.allclose(J[9:], 0.0))
print("div J = 0 exactly, so NO div-J correction is added to the drift.")
pd.DataFrame(np.round(J, 4), index=data.parameter_names, columns=data.parameter_names)

## 6. The projection onto $K$

Feasible points are returned unchanged; otherwise the scalar KKT multiplier $\eta$ is found by an outer root solve around the $d$ decoupled coordinate equations.

In [ ]:
rng = np.random.default_rng(3)
records = []
for _ in range(5):
    y = w_map + 1.5 * rng.standard_normal(d)
    result = projection.project_K(y, p_constraint, epsilon_constraint, Lambda_constraint)
    residual = projection.kkt_residual(y, result, p_constraint, epsilon_constraint,
                                       Lambda_constraint)
    records.append({"g(y)": result.g_before, "projected": result.projected,
                    "g(z)": result.g_after, "distance": result.distance,
                    "eta": result.eta, **residual})
pd.DataFrame(records)

## 7. Run the full study

`run_experiment` executes the pilot, freezes $\Lambda_{\text{constraint}}$, finds the constrained MAP, calibrates one common step size at the largest $\alpha$, runs the primary $\alpha$ comparison with common random numbers, performs the automatic checks, the sensitivity studies and the predictive evaluation, and writes every artefact.

*(This is the long cell: with the production settings above it takes on the order of ten minutes.)*

In [ ]:
results = run_experiment(config, verbose=True)
output_dir = Path(results["output_dir"])

## 8. Automatic checks (section 14)

In [ ]:
checks = results["checks"]
assert checks["passed"].all(), checks.loc[~checks["passed"]]
display(checks.style.hide(axis="index"))

## 9. Diagnostics by $\alpha$

In [ ]:
display(results["diagnostics"][[
    "label", "min_ess", "median_ess", "min_ess_per_second", "max_split_rhat",
    "max_tau_int", "mean_squared_jumping_distance", "projection_frequency",
    "mean_projection_distance", "runtime_seconds", "gradient_evaluations"]])
display(results["coefficients"].head(22))
if results["warnings"]:
    print("\n".join("! " + w for w in results["warnings"]))
else:
    print("no warnings raised")

## 10. Posterior predictive performance (held-out test split)

MAGIC is imbalanced, so balanced accuracy, ROC-AUC, sensitivity, specificity and calibration are read before plain accuracy. $\alpha$ was chosen from training diagnostics only.

In [ ]:
display(results["predictive"][[
    "label", "accuracy", "balanced_accuracy", "roc_auc", "log_loss", "brier_score",
    "sensitivity_recall_positive", "specificity_recall_negative", "precision", "f1"]])
display(pred.confusion_matrix_frame(results["reports"][0]))

## 11. Figures

Every figure is written to `figures/` as both PDF and 300-dpi PNG.

In [ ]:
from IPython.display import Image, display as show

for name in sorted(p.name for p in (output_dir / "figures").glob("*.png")):
    print(name)
    show(Image(filename=str(output_dir / "figures" / name), width=900))

## 12. Sensitivity studies

*Constraint radius*: different thresholds define **different truncated posteriors**, so a shift in the summaries is expected behaviour, not a bug. *Step size*: $h$, $h/2$, $h/4$ target the same distribution and differ only through the $O(h)$ discretisation bias of the unadjusted scheme; agreement between $h$ and $h/2$ is the evidence that $h$ is small enough.

In [ ]:
if results["constraint_sensitivity"]:
    display(diag.diagnostics_table(results["constraint_sensitivity"])[
        ["label", "alpha", "Lambda_constraint", "min_ess", "max_split_rhat",
         "projection_frequency", "mean_projection_distance"]])
if results["step_sensitivity"]:
    display(diag.diagnostics_table(results["step_sensitivity"])[
        ["label", "alpha", "step_size", "min_ess", "max_split_rhat",
         "mean_squared_jumping_distance"]])
if results["extension"]:
    print("overlapping-triple extension (labelled sensitivity experiment):")
    display(diag.diagnostics_table(results["extension"])[
        ["label", "alpha", "min_ess", "median_ess", "max_split_rhat",
         "mean_J_operator_norm", "mean_squared_jumping_distance"]])

## 13. Written artefacts

In [ ]:
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(output_dir)!s:<55} {path.stat().st_size / 1024:8.1f} KiB")
print()
print((output_dir / "summary.md").read_text()[:2500])